# Tutorial 4: Training

## Configuration, Optimization, and Convergence

---

This notebook walks through the training pipeline — from YAML configuration to checkpoint. We'll examine every design choice: learning rate scheduling, mixed-precision training, gradient clipping, and early stopping.

### What you'll learn

1. How to read and customise YAML configs
2. The loss function: negative log-probability under the flow
3. Learning rate schedule: linear warmup + cosine annealing
4. Automatic mixed precision (AMP) for GPU efficiency
5. The early stopping + checkpointing strategy
6. How to run a mini training loop and interpret the loss curve

### Modules covered

| Module | Purpose |
|--------|---------|
| `src.train` | Full training loop |
| `src.utils` | Config loading, seeding, device selection |
| `src.plots` | Training curve visualisation |

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
import json

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

SEED = 42
torch.manual_seed(SEED)

---
## 1. Configuration Walkthrough

All hyperparameters live in a single YAML file. Let's inspect the full v5 configuration:

In [ ]:
cfg_path = os.path.join(PROJECT_ROOT, 'configs', 'transformer_v5.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Pretty-print each section
for section in ['data', 'model', 'training', 'eval']:
    print(f'\n=== {section} ===')
    for k, v in cfg[section].items():
        print(f'  {k}: {v}')

# v5 prior is nested: global (4D) + white_noise (3D per backend)
print(f'\n=== prior ===')
for sub, params in cfg['prior'].items():
    print(f'  {sub}:')
    for k, v in params.items():
        print(f'    {k}: {v}')

print(f'\noutput_dir: {cfg["output_dir"]}')

In [ ]:
# Also look at the smoke config (for fast testing)
smoke_path = os.path.join(PROJECT_ROOT, 'configs', 'smoke_v5.yaml')
if os.path.exists(smoke_path):
    with open(smoke_path) as f:
        smoke_cfg = yaml.safe_load(f)
    
    print('Smoke config differences from full v5:')
    for section in ['data', 'model', 'training']:
        full_sec = cfg.get(section, {})
        smoke_sec = smoke_cfg.get(section, {})
        for k in full_sec:
            v_full = full_sec[k]
            v_smoke = smoke_sec.get(k, v_full)
            if v_full != v_smoke:
                print(f'  {section}.{k}: {v_full} → {v_smoke}')

---
## 2. The Loss Function

The training objective is **neural posterior estimation (NPE)**: maximise the log-probability of the true parameters $\boldsymbol{\theta}$ under the learned conditional density:

$$
\mathcal{L} = -\mathbb{E}_{p(\boldsymbol{\theta}, \mathbf{x})}\left[ \log q_\phi(\boldsymbol{\theta} \mid \mathbf{x}) \right]
$$

This is simply the cross-entropy between the true posterior and the learned approximation. At optimum, $q_\phi = p(\boldsymbol{\theta} \mid \mathbf{x})$ exactly.

In practice, each batch provides a Monte Carlo estimate:
$$
\hat{\mathcal{L}} = -\frac{1}{B} \sum_{i=1}^{B} \log q_\phi(\boldsymbol{\theta}_i \mid \mathbf{x}_i)
$$

In [ ]:
from src.priors import FactorizedPrior
from src.dataset import PulsarDataset
from src.collate import collate_fn
from src.models.model_wrappers import build_model, FactorizedNPEModel
from torch.utils.data import DataLoader

# Build a small model for demonstration
prior = FactorizedPrior(cfg['prior']['global'], cfg['prior']['white_noise'], n_backends_max=4)
model = build_model('transformer', cfg)

# Small dataset — factorized=True produces theta_global and theta_wn keys
ds = PulsarDataset(256, prior, cfg['data'], seed=42,
                   masking_severity=0.5, augment=True, use_sobol=False, factorized=True)
loader = DataLoader(ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
batch = next(iter(loader))

# Compute factorized NPE loss
loss = model(batch)
print(f'Factorized NPE loss = {loss.item():.4f}')
print(f'  global flow (4D):     {model._last_global_loss:.4f}')
print(f'  wn flow (3D/backend): {model._last_wn_loss:.4f}')
print(f'  combined = global + {cfg["model"]["wn_loss_weight"]} × wn')
print(f'\nBatch keys: {list(batch.keys())}')
print(f'  theta_global: {batch["theta_global"].shape}  (B, 4)')
print(f'  theta_wn:     {batch["theta_wn"].shape}  (B, n_backends_max, 3)')

---
## 3. Learning Rate Schedule

The v5 config uses:

1. **Linear warmup** for the first 10% of epochs (start at 0.01× LR, ramp to full LR)
2. **Cosine annealing** for the remaining 90% (smooth decay to 0)

This is implemented via `torch.optim.lr_scheduler.SequentialLR` combining `LinearLR` and `CosineAnnealingLR`.

The peak LR is 3×10⁻⁴ — lower than earlier versions. Combined with the longer warmup (20 epochs for the 200-epoch schedule), this prevents NSF spline knots from drifting into numerically unstable regions during the early high-LR phase.

In [ ]:
# Simulate the LR schedule
tcfg = cfg['training']
epochs = tcfg['epochs']
warmup_epochs = max(1, int(tcfg['warmup_fraction'] * epochs))

# Create optimizer + scheduler (same as train.py)
optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg['lr'],
                               weight_decay=tcfg['weight_decay'])
warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, total_iters=warmup_epochs)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs - warmup_epochs)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, [warmup, cosine], milestones=[warmup_epochs])

# Record LR at each epoch
lrs = []
for ep in range(epochs):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, epochs + 1), lrs, lw=1.5)
ax.axvline(warmup_epochs, color='red', ls='--', lw=1, label=f'Warmup ends (epoch {warmup_epochs})')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning rate')
ax.set_title(f'LR Schedule: {warmup_epochs} warmup + {epochs - warmup_epochs} cosine')
ax.legend()
ax.set_yscale('log')
ax.set_xlim(1, epochs)
fig.tight_layout()
plt.show()

print(f'Peak LR: {max(lrs):.2e} at epoch {lrs.index(max(lrs))+1}')
print(f'Final LR: {lrs[-1]:.2e}')

---
## 4. Mixed Precision & Gradient Clipping

Two techniques accelerate and stabilise training:

### Automatic Mixed Precision (AMP)
- The encoder runs in float16 on CUDA (≈2× faster with modern GPUs)
- The flow is **forced back to float32** via explicit `torch.autocast(..., enabled=False)` — spline flows are numerically unstable in float16
- `torch.amp.GradScaler` prevents gradient underflow in float16

### Gradient Clipping
- `grad_clip=1.0` — max gradient norm
- Prevents spikes from occasional pathological batches (outlier likelihoods)

In [ ]:
# Demonstrate the AMP flow bypass in FactorizedNPEModel.forward()
import inspect

# Show the key lines from FactorizedNPEModel.forward()
source = inspect.getsource(model.forward)
for line in source.split('\n'):
    if 'autocast' in line or 'float()' in line or 'log_prob' in line:
        print(line.strip())

print('\nKey: both flows (global + WN) run in float32 even when AMP autocasts the encoder to float16.')
print('Rational-quadratic splines involve log/exp operations that lose precision in float16,')
print('leading to NaN gradients without this bypass.')

---
## 5. Training Loop Anatomy

The `train_one_epoch` function follows a standard pattern. Let's trace through it:

In [ ]:
from src.train import train_one_epoch, validate

# Reset model and optimizer for a clean demo
torch.manual_seed(SEED)
model = build_model('transformer', cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

device = torch.device('cpu')

# Run a few training steps on our small dataset
n_demo_epochs = 5
train_losses = []

print(f'Training {n_demo_epochs} epochs on {len(ds)} samples (batch_size=16)...')
print(f'{"Epoch":>5s}  {"Train Loss":>10s}')

for ep in range(1, n_demo_epochs + 1):
    tl = train_one_epoch(model, loader, optimizer, device, grad_clip=1.0)
    train_losses.append(tl)
    print(f'{ep:5d}  {tl:10.4f}')

print(f'\nLoss decreased from {train_losses[0]:.4f} to {train_losses[-1]:.4f}')
print(f'Even a few epochs show the model is learning.')

---
## 6. Early Stopping & Checkpointing

The training loop monitors validation loss:

- **Best model saved** whenever `val_loss < best_val_loss`
- **Patience counter** increments when no improvement
- **Training stops** when patience reaches `patience=20` epochs

The checkpoint includes the full model state, optimizer state, config, and epoch number — enabling both evaluation and training resumption.

In [ ]:
# Inspect a saved checkpoint (if available)
ckpt_path = os.path.join(PROJECT_ROOT, 'outputs', 'v5', 'transformer', 'best_model.pt')

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'Checkpoint keys: {list(ckpt.keys())}')
    print(f'  epoch:      {ckpt["epoch"]}')
    print(f'  val_loss:   {ckpt["val_loss"]:.4f}')
    print(f'  model_type: {ckpt["model_type"]}')
    
    # Config stored in checkpoint
    ckpt_cfg = ckpt['config']
    print(f'  config[training][epochs]: {ckpt_cfg["training"]["epochs"]}')
    print(f'  config[training][lr]:     {ckpt_cfg["training"]["lr"]}')
else:
    print(f'No checkpoint found at {ckpt_path}')
    print('Run training first:')
    print('  python -m src.train --config configs/transformer_v5.yaml --model transformer')

In [ ]:
# Inspect training metrics (if available)
metrics_path = os.path.join(PROJECT_ROOT, 'outputs', 'v5', 'transformer', 'train_metrics.json')

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)
    
    print(f'Training summary:')
    print(f'  Model: {metrics["model_type"]}')
    print(f'  Parameters: {metrics["n_params"]:,}')
    print(f'  Final epoch: {metrics["final_epoch"]}')
    print(f'  Best val loss: {metrics["best_val_loss"]:.4f}')
    
    # v5 logs per-component losses (global + WN)
    has_factorized = 'train_global_losses' in metrics
    n_panels = 3 if has_factorized else 1
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 4))
    if n_panels == 1:
        axes = [axes]
    
    axes[0].plot(metrics['train_losses'], label='Train', alpha=0.8)
    axes[0].plot(metrics['val_losses'], label='Validation', alpha=0.8)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Total Loss — {metrics["model_type"]}')
    axes[0].legend()
    
    if has_factorized:
        axes[1].plot(metrics['train_global_losses'], label='Train', alpha=0.8)
        axes[1].plot(metrics['val_global_losses'], label='Validation', alpha=0.8)
        axes[1].set_xlabel('Epoch')
        axes[1].set_title('Global Flow (4D: A_red, γ_red, A_dm, γ_dm)')
        axes[1].legend()
        
        axes[2].plot(metrics['train_wn_losses'], label='Train', alpha=0.8)
        axes[2].plot(metrics['val_wn_losses'], label='Validation', alpha=0.8)
        axes[2].set_xlabel('Epoch')
        axes[2].set_title('WN Flow (3D per backend)')
        axes[2].legend()
    
    for ax in axes:
        ax.set_xlim(0, len(metrics['train_losses']))
    fig.tight_layout()
    plt.show()
else:
    print('No training metrics found. Run training to generate them.')

---
## 7. Running Training

To launch a full training run from the command line:

```bash
# Full training (GPU recommended)
python -m src.train --config configs/transformer_v5.yaml --model transformer \
    --log-file outputs/v5_train.log

# Quick smoke test
python -m src.train --config configs/smoke_v5.yaml --model transformer

# LSTM baseline
python -m src.train --config configs/transformer_v5.yaml --model lstm
```

Key outputs:
- `outputs/v5/transformer/best_model.pt` — best checkpoint
- `outputs/v5/transformer/train_metrics.json` — loss curves & metadata (global + WN components)
- `outputs/v5/transformer/training_curves.png` — plot

---
## Summary

The v5 training pipeline is designed for stability with a factorized posterior:

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Loss | global NLL + 0.3 × WN NLL | Factorized NPE; WN weight balances backend count |
| Optimizer | AdamW (per-group weight decay) | 5×10⁻⁴ encoder, 2×10⁻³ flows |
| LR schedule | 10% warmup + 90% cosine | Longer warmup for NSF stability |
| Peak LR | 3×10⁻⁴ | Lower than v4 to prevent spline drift |
| AMP | Float16 encoder, float32 flows | Speed without spline instability |
| Grad clip | Norm ≤ 1.0 | Tighter constraint for NSF stability |
| Early stopping | Patience = 40 | More tolerance for per-epoch variance |
| Epoch reseeding | `reseed_per_epoch: true` | New (θ, x) pairs each epoch — prevents flow memorisation |

### Key takeaways

1. The **factorized loss** logs both global (4D) and WN (3D) components separately — monitor both for convergence
2. **10% warmup + grad_clip=1.0** are critical: spline knots collapse without adequate warm-up and gradient control
3. **AMP with float32 flows** nearly doubles throughput on modern GPUs
4. **Epoch reseeding** generates fresh simulations every epoch, eliminating the training-set memorisation risk
5. **Per-group weight decay** constrains flow parameters 4× more aggressively than the encoder
6. Checkpoints store the full config, enabling exact reproduction

**Next**: [Tutorial 5 — Evaluation](05_evaluation.ipynb) covers posterior quality metrics, calibration diagnostics, and model comparison.